# Module 1 — LLM Fundamentals: OpenAI Practical Experiments

A six-day companion to `Module-1-LLM-Fundamentals-Detailed.md`. Learn by running, inspecting, changing, and recording results. This notebook uses the **OpenAI Python SDK and Responses API**. Run setup first, then the units in order. Python 3.10+ is recommended.

**What you will practice:** repeated model calls; tokens, cost, context and latency; images, PDFs and recorded audio; output limits and temperature; failure analysis; and model selection with a labeled dataset.

**Cost and privacy:** Cells marked ▶ make paid API calls, and optional loops may make many calls. Work with the small examples first, check [current prices](https://developers.openai.com/api/docs/pricing), and set a budget for your account. Use synthetic or authorized data only; images, PDFs, and audio are sent to OpenAI. Notebook outputs can contain your data: clear them before sharing. Results vary; no example claims a model is always correct.

**Source corrections:** The module includes older SDK calls, illustrative prices and some overgeneralizations. This notebook uses current SDK patterns, measured token counts, and explicitly specified hypothetical rates. A model can be combined with tools or retrieval; without those, a plain request does not automatically verify current facts. Streaming improves perceived responsiveness, not token pricing. A stream chunk is not necessarily one token.


## 0. Environment setup

1. Select your virtual environment as the Jupyter kernel in VS Code.
2. Install the packages with the next cell. If needed, restart the kernel afterwards.
3. Put `OPENAI_API_KEY=your-key` in a local `.env` in this folder or a parent folder, or set it in the environment **before** launching VS Code. Never paste the secret in a notebook. Ensure `.env` is ignored by Git.
4. A model must be available to your OpenAI project. Override the default with `OPENAI_MODEL` in your environment. Check supported inputs, parameters, snapshots, and prices in the [model catalog](https://developers.openai.com/api/docs/models).

The setup lets you run offline exercises without a key. Calling `require_client()` gives a clear message if the current Jupyter kernel cannot see it.


In [1]:
%pip install -U openai python-dotenv tiktoken matplotlib ipykernel


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 7.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 7.7 MB/s  0:00:01 eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 6.2 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: httpcore2
    Found existing installation: httpcore2 2.10.0
    Uninstalling httpcore2-2.10.0:
      Successfully uninstalled httpcore2-2.10.0
  Attempting uninstall: httpx2━━━━╸━━━━━━━━━━━━━ 6/9 [matplotlib]
    Found existing installation: httpx2 2.10.090m━━━━━━━━━━━━━ 6/9 [matplotlib]
    Uninstalling httpx2-2.10.0:╸━━━━━━━━━━━━━ 6/9 [matplotlib]
      Successfully uninstalled httpx2-2.10.0━━━━━━━━━━━━━ 6/9 [matplotlib]
  Attempting uninstall: openai╸━━━━━━━━━━━━━ 6/9 [matplotlib]
    Found existing installation: openai 3.2.0━━━━━━━━━━━━━ 6/9 [matplotlib]
    Uninstalling openai-3.2.0:━━━━━━━━━━━━━╸━━━━ 8/9 [openai]
      Successfully uninstalled openai-3.2.0╸━━━━ 8/9 [openai]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os, json, time, base64, statistics, re
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
import tiktoken
from openai import OpenAI

# VS Code may open the notebook from either workspace root or the notebook folder.
for env_path in [Path.cwd() / ".env", Path.cwd() / "GenAI/Module_1/.env"]:
    if env_path.is_file(): load_dotenv(env_path, override=False)
found_env = find_dotenv(usecwd=True)
if found_env: load_dotenv(found_env, override=False)
MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
print("Model:", MODEL, "| API key available to this kernel:", bool(os.getenv("OPENAI_API_KEY")))
def require_client():
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Set OPENAI_API_KEY in a private .env or the kernel environment, then rerun setup.")
    return OpenAI(timeout=60.0, max_retries=2)
def ask(prompt, model=MODEL, max_output_tokens=500, **kwargs):
    start = time.perf_counter()
    response = require_client().responses.create(model=model, input=prompt, max_output_tokens=max_output_tokens, **kwargs)
    return response, time.perf_counter() - start
def show(response, seconds):
    print(response.output_text)
    print("Status:", response.status, "| incomplete:", response.incomplete_details,
          "| usage:", response.usage, "| seconds:", round(seconds, 2))


Model: gpt-6-luna | API key available to this kernel: True


## 1.1 — What an LLM does (Day 1)

A language model assigns probabilities to possible next tokens, conditioned on the context, and generates a sequence. The model's responses can vary between runs. Pretraining and later instruction training shape behavior. Parameters are learned values, not a direct measure of quality on your task.

**Experiment:** Make the same open-ended request five times without changing settings. Compare common ideas and differences. Identical answers are also an observation; a handful of samples cannot prove a distribution.


In [ ]:
# ▶ 5 paid requests. Set runs=2 for a quick start.
prompt = "The future of AI in the next five years will be marked by... Complete in two sentences."
def repeat_prompt(prompt, runs=5, **kwargs):
    responses = []
    for i in range(runs):
        response, elapsed = ask(prompt, **kwargs)
        responses.append(response.output_text)
        print(f"Run {i+1}: {response.output_text}\n  status={response.status}, elapsed={elapsed:.2f}s")
    return responses
# repeated_answers = repeat_prompt(prompt)


**Try it:** Record themes, word choice and whether any factual claims are verifiable. Change only the prompt and compare. Avoid treating fluent output as proof. Temperature zero, when supported, reduces sampling variability but does not guarantee identical outputs.


## 1.2 — Tokens, cost, context and latency (Day 2)

A token is a model-specific piece of text; it is not necessarily a word or character. Exact tokenization depends on the encoding. A local text count excludes message framing, tool schemas, images and other overhead. The API's returned usage is the better measure of a particular request.


In [ ]:
# Offline: inspect token IDs and decoded bytes with an explicit encoding.
# o200k_base is an example encoding; confirm the mapping for your selected model.
try:
    encoding = tiktoken.encoding_for_model(MODEL)
    print("Model mapping:", encoding.name)
except KeyError:
    encoding = tiktoken.get_encoding("o200k_base")
    print("Unknown model mapping; using o200k_base for illustration only.")
def inspect_tokens(text):
    ids = encoding.encode(text)
    print("Text:", repr(text), "| characters:", len(text), "| tokens:", len(ids))
    print("IDs:", ids)
    print("Bytes per token:", [encoding.decode_single_token_bytes(i) for i in ids])
inspect_tokens("Hello, how are you today?")


In [ ]:
# Offline: compare equivalent ideas in multiple formats; these counts are empirical.
samples = {
    "English": "The weather is pleasant today. We will visit the market and return home before dinner.",
    "Hindi": "आज मौसम सुहावना है। हम बाजार जाएंगे और रात के खाने से पहले घर लौटेंगे।",
    "Mixed": "आज weather अच्छा है। We will बाजार जाएंगे और dinner से पहले घर लौटेंगे।",
    "Python": "def total(prices):\n    return sum(price for price in prices if price > 0)\n",
    "Numbers": "202-555-0101, 202-555-0102, 202-555-0103, 202-555-0104, 202-555-0105, 202-555-0106, 202-555-0107, 202-555-0108, 202-555-0109, 202-555-0110",
    "JSON": json.dumps({"name": "Example User", "city": "Surat", "orders": [12, 35], "active": True})
}
for label, sample in samples.items():
    print(f"{label:8} characters={len(sample):3} tokens={len(encoding.encode(sample)):3}")


**Try it:** Replace the first two samples with your own roughly 100-word translation pair and add a ten-line code snippet. Compare tokens, not just word count. Languages and encodings vary, so the source's fixed Hindi/English ratios are not universal.


In [ ]:
# Offline: enter verified current rates to estimate your own text-only usage.
def cost_usd(input_tokens, output_tokens, input_rate, output_rate):
    """Rates are USD per million tokens; excludes discounts and other charges."""
    return (input_tokens * input_rate + output_tokens * output_rate) / 1_000_000
users_per_hour, input_per_user, output_per_user = 100, 1000, 200
# Source's numbers are hypothetical rates, not current model pricing.
hourly = cost_usd(users_per_hour*input_per_user, users_per_hour*output_per_user, 5, 15)
print("Hypothetical hourly:", hourly, "USD; daily:", hourly*24, "; 30-day:", hourly*24*30)
assert abs(hourly - 0.80) < 1e-9


The source's Day 2 example has an arithmetic error: at its hypothetical $5/$15 per million rates, 100,000 input plus 20,000 output tokens costs **$0.80/hour**, not $0.0008/hour. At 24 hours/day, that is $19.20/day and $576 per 30 days. Cached input, reasoning, image/audio billing and tools can change actual charges; check the provider's pricing and usage.


In [ ]:
# ▶ Compare local text estimate with returned input/output usage.
sample_prompt = "Explain context windows in two sentences."
print("Local plain-text estimate:", len(encoding.encode(sample_prompt)))
response, elapsed = ask(sample_prompt)
show(response, elapsed)


### Context, knowledge, and application memory

Context is the material available for the current request. Model knowledge comes from training. Application memory is information your application stores and supplies or links to a conversation. The Responses API also supports server-managed conversation state; in the experiment below each call is deliberately independent unless we supply the previous text ourselves. Allow space for input, output and model-specific reasoning tokens within limits.


In [ ]:
# ▶ Three independent calls: no stored conversation is referenced.
first, _ = ask("Remember this for the next question: my favorite color is teal. Reply OK.")
without_context, _ = ask("What is my favorite color? If unknown, say unknown.")
with_context, _ = ask("The user's favorite color is teal. What is the user's favorite color?")
print("First:", first.output_text)
print("Without context:", without_context.output_text)
print("With context:", with_context.output_text)


In [ ]:
# Offline context budget planning. Example assumes an 8,000-token combined budget.
context_limit = 8000
budget = {"instructions": 300, "question": 200, "retrieved_documents": 6000, "reserved_output": 500, "safety_margin": 1000}
print(budget, "total:", sum(budget.values()), "remaining:", context_limit-sum(budget.values()))
assert sum(budget.values()) <= context_limit


In [ ]:
# ▶ Measure total time and first visible text on separate requests.
latency_prompt = "Explain the history of AI in three short paragraphs."
response, total = ask(latency_prompt, max_output_tokens=800)
print("Non-streaming total seconds:", round(total, 2), "usage:", response.usage)
start = time.perf_counter(); first_text = None; parts = []; final_response = None
stream = require_client().responses.create(model=MODEL, input=latency_prompt, max_output_tokens=800, stream=True)
for event in stream:
    if event.type == "response.output_text.delta":
        if first_text is None: first_text = time.perf_counter()-start
        parts.append(event.delta)
        print(event.delta, end="", flush=True)
    elif event.type == "response.completed": final_response = event.response
    elif event.type in ("response.failed", "error"): print("\nStream error:", event.type)
stream_total = time.perf_counter()-start
print("\nFirst visible text (seconds):", first_text, "| total:", round(stream_total, 2))
print("Final usage if available:", final_response.usage if final_response else None)
print("Stream deltas:", len(parts), "(not a token count)")


**Try it:** Repeat short and long prompts several times. Record first visible text, total time, output-token usage and status. Throughput measured as output tokens / total time includes waiting and possibly reasoning; it is not pure decoding speed. Longer context may cost more and may distract the model: try placing a relevant fact near the beginning, middle, and end among harmless filler and score repeated trials.


## 1.3 — Images, PDFs and recorded audio (Day 3)

Use a vision-capable model and only synthetic or authorized files. Vision can miss small text, count objects incorrectly, or misread charts. Image/PDF usage is not estimated by the plain-text tokenizer above. Check the [image guide](https://developers.openai.com/api/docs/guides/images-vision) and [file input guide](https://developers.openai.com/api/docs/guides/file-inputs) for supported formats and limits.

The experiment creates a fictional chart locally, then compares sending its image versus sending its known data. The image and text calls cost money; the file generation is local.


In [ ]:
# Offline: create an image with known ground truth.
import matplotlib.pyplot as plt
sales = {"North": 125, "South": 98, "East": 156, "West": 142}  # thousand USD
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(sales.keys(), sales.values(), color="steelblue")
ax.set(title="Fictional Q4 Sales", ylabel="Thousands of USD")
for i, value in enumerate(sales.values()): ax.text(i, value+2, str(value), ha="center")
fig.tight_layout()
chart_path = Path("module1_demo_chart.png")
fig.savefig(chart_path, dpi=150)
plt.show()
print("Created:", chart_path.resolve())


In [ ]:
# ▶ Send image bytes as a data URL, then compare with the underlying text.
image_data = base64.b64encode(chart_path.read_bytes()).decode("ascii")
question = "List every region and its value, and identify the highest. State units."
start = time.perf_counter()
image_response = require_client().responses.create(model=MODEL, max_output_tokens=500, input=[{
    "role": "user", "content": [
        {"type": "input_text", "text": question},
        {"type": "input_image", "image_url": "data:image/png;base64," + image_data},
    ]}])
image_seconds = time.perf_counter()-start
text_response, text_seconds = ask(question + "\nFictional sales, thousands of USD: " + json.dumps(sales))
print("Image result:"); show(image_response, image_seconds)
print("\nText result:"); show(text_response, text_seconds)
print("Ground truth:", sales)


**Try it:** Check every number manually, compare usage and latency, and resize or obscure a label. Do not assume either method is always cheaper or more accurate. A digitally generated PDF can contain extractable text; a scanned PDF can require vision/OCR. Verify extracted numbers and layout against the original.


In [ ]:
# ▶ Optional PDF experiment: set an existing, non-sensitive PDF path before running.
# See the file-input documentation for limits and model compatibility.
def inspect_pdf(pdf_path):
    pdf_path = Path(pdf_path)
    encoded = base64.b64encode(pdf_path.read_bytes()).decode("ascii")
    response = require_client().responses.create(model=MODEL, max_output_tokens=500, input=[{
        "role": "user", "content": [
            {"type": "input_file", "filename": pdf_path.name, "file_data": "data:application/pdf;base64," + encoded},
            {"type": "input_text", "text": "Summarize the document and identify uncertain or unreadable parts."},
        ]}])
    print(response.output_text, "\nUsage:", response.usage, "Status:", response.status)
# inspect_pdf("your-synthetic-document.pdf")


In [ ]:
# ▶ Optional recorded audio experiment; use an authorized short file in a supported format.
# The speech-to-text guide documents file limits, formats, and current model names.
def transcribe_audio(audio_path):
    with open(audio_path, "rb") as audio_file:
        result = require_client().audio.transcriptions.create(model="gpt-transcribe", file=audio_file)
    print(result.text)
    return result.text
# transcript = transcribe_audio("your-synthetic-recording.wav")


**Try it:** For audio, compare the transcript to a written ground truth, especially names, numbers, background noise and code-switching. Review sensitive or consequential content manually. Recorded-file transcription and live audio are separate workflows; this example uses a completed file. [Speech-to-text documentation](https://developers.openai.com/api/docs/guides/speech-to-text).


## 1.4 — Controlling output (Day 4)

Output limits affect latency, cost and whether a response is complete. Some reasoning models use part of the output budget for reasoning. Inspect `status` and `incomplete_details`. Parameter support, including temperature and reasoning effort, is model-specific: check the selected model's page before sending them.


In [ ]:
# ▶ Compare small and larger output budgets; small budgets may return no visible answer on reasoning models.
for limit in [32, 500]:
    response, elapsed = ask("Explain how a library organizes books, with three examples.", max_output_tokens=limit)
    print("\nOutput limit:", limit); show(response, elapsed)


In [ ]:
# ▶ Optional temperature study: only run with a model that documents support for temperature.
# Set TEMPERATURE_MODEL to that model's ID first; API errors indicate unsupported settings.
tasks = {
    "extraction": "Extract the numbers only: Annual revenue is $2.5M, up 15%.",
    "creative": "In two sentences describe what it feels like to be a library book.",
    "reasoning": "All roses are flowers and all flowers bloom. Do all roses bloom? Explain briefly."
}
def temperature_study(model_name, temperatures=(0, 0.5, 1.0, 1.5), repeats=2):
    for task, task_prompt in tasks.items():
        for temperature in temperatures:
            for run in range(repeats):
                try:
                    r, elapsed = ask(task_prompt, model=model_name, temperature=temperature)
                    print(task, temperature, run+1, r.output_text, "status:", r.status, "s:", round(elapsed, 2))
                except Exception as exc:
                    print(task, temperature, "failed:", type(exc).__name__, str(exc)[:250])
                    break
# temperature_study(os.environ["TEMPERATURE_MODEL"])  # up to 24 paid requests


**Try it:** Compare accuracy and variation across repeated outputs, not just one answer per setting. Low temperature does not guarantee correctness or reproducibility; higher values do not guarantee creativity. If the model supports reasoning-effort configuration, compare documented levels using the same reasoning examples and record quality, usage and latency.


## 1.5 — Where LLMs fail (Day 5)

Model confidence and fluency are not evidence. Test factual knowledge, impossible questions, arithmetic, constraints, and reasoning. For a date-specific fact, consult a reliable source; do not score a guess based on its tone. A request to invent a language is a creative task, not necessarily an impossible one.


In [ ]:
failure_cases = [
 ("What is the capital of Portugal?", "Known answer: Lisbon."),
 ("What was the population of London in 1850?", "Needs a cited, independently verified estimate and definition of London."),
 ("What did I eat for breakfast?", "Unknown without context; should not invent it."),
 ("How many people have visited Mars?", "Verify with a reliable current source; do not infer from confidence."),
 ("What is 47 multiplied by 89?", f"Use Python as ground truth: {47*89}."),
 ("Invent a word for hello in an invented language, labeling it as invented.", "Creative output is acceptable if labeled."),
 ("Who is the CEO of the imaginary company Acme that I just made up?", "Unknown without context."),
 ("All cats are animals. Fluffy is a cat. Is Fluffy an animal?", "Yes, under the premises."),
 ("Write exactly a three-word sentence.", "Count words and examine punctuation."),
 ("What color will the sky be at my location next Tuesday?", "Missing location and future conditions; avoid false certainty."),
]
# ▶ Optional: 10 calls. Score manually against the stated criteria.
# for question, rubric in failure_cases:
#     response, _ = ask(question)
#     print("Q:", question, "\nA:", response.output_text, "\nRubric:", rubric, "\n")


### Measure task accuracy on a labeled dataset

Task: classify synthetic support messages into **billing**, **technical**, **feature**, or **other**. The 20 labels below are a starting rubric. Review them before testing; some real tickets belong to more than one category. Count failed or invalid responses as unsuccessful, and inspect them separately. A 20-item dataset is a learning sample, not a reliable production benchmark.


In [ ]:
cases = [
 ("I was charged twice for my plan.", "billing"), ("Please send my invoice.", "billing"),
 ("Why did my subscription price increase?", "billing"), ("My refund has not arrived.", "billing"),
 ("The payment was declined but money was deducted.", "billing"),
 ("The app crashes when I open settings.", "technical"), ("Nobody can sign in; login shows an error.", "technical"),
 ("My upload freezes at 50 percent.", "technical"), ("The website does not load in my browser.", "technical"),
 ("Password reset link returns an error.", "technical"),
 ("Please add dark mode.", "feature"), ("Can you support CSV exports?", "feature"),
 ("I would like a calendar integration.", "feature"), ("Could you add multi-language support?", "feature"),
 ("Please make the dashboard customizable.", "feature"),
 ("What are your office hours?", "other"), ("Where can I read the terms of service?", "other"),
 ("Thank you for the helpful response.", "other"), ("How can I contact your team?", "other"),
 ("What does your company do?", "other"),
]
assert len(cases) == 20
LABELS = {"billing", "technical", "feature", "other"}
def evaluate_model(model_name, examples, rates=None):
    rows = []
    for index, (message, expected) in enumerate(examples, 1):
        prompt = ("Classify this support message. Return exactly one lowercase label: "
                  "billing, technical, feature, or other. If mixed, choose the main request.\nMessage: " + message)
        start = time.perf_counter()
        try:
            response = require_client().responses.create(model=model_name, input=prompt, max_output_tokens=200)
            predicted = response.output_text.strip().lower()
            usage = response.usage
            input_tokens = usage.input_tokens if usage else None
            output_tokens = usage.output_tokens if usage else None
            cost = (cost_usd(input_tokens, output_tokens, *rates)
                    if rates is not None and input_tokens is not None and output_tokens is not None else None)
            rows.append({"id": index, "expected": expected, "predicted": predicted,
                         "valid": predicted in LABELS and response.status == "completed",
                         "correct": predicted == expected and response.status == "completed",
                         "status": response.status, "input_tokens": input_tokens, "output_tokens": output_tokens,
                         "cost_usd": cost, "seconds": time.perf_counter()-start, "error": None})
        except Exception as exc:
            rows.append({"id": index, "expected": expected, "predicted": None, "valid": False, "correct": False,
                         "status": "error", "input_tokens": None, "output_tokens": None, "cost_usd": None,
                         "seconds": time.perf_counter()-start, "error": type(exc).__name__})
    return rows
def summarize_evaluation(rows):
    if not rows: print("No examples."); return
    known_costs = [r["cost_usd"] for r in rows if r["cost_usd"] is not None]
    print("Accuracy:", sum(r["correct"] for r in rows), "/", len(rows),
          "| invalid or failed:", sum(not r["valid"] for r in rows))
    print("Mean seconds:", round(statistics.mean(r["seconds"] for r in rows), 2),
          "| max seconds:", round(max(r["seconds"] for r in rows), 2))
    print("Known estimated USD:", round(sum(known_costs), 6), "| unknown cost requests:", len(rows)-len(known_costs))
    print("Failures:")
    for row in rows:
        if not row["correct"]: print(row)
# ▶ Optional: 20 paid requests; try cases[:3] first.
# results = evaluate_model(MODEL, cases[:3])
# summarize_evaluation(results)


**Try it:** Classify all 20 examples, manually inspect the failures, then revise the prompt. Keep a separate held-out test set to see whether the revision generalizes. Add edge cases such as mixed intents and ambiguous messages. Calculate per-category accuracy; a high overall score can conceal a weak category.


## 1.6 — Choose a model from evidence (Day 6)

Choose a task and define thresholds before testing: quality, cost, latency, context, modality, and privacy. Model aliases and availability can change; if a supported snapshot is available, record or pin it and rerun evaluations on upgrades. Hosted APIs and open-weight/self-hosted options differ in data handling, infrastructure and license obligations; neither is automatically cheaper or more capable for every workload.


In [ ]:
# ▶ Model comparison: choose accessible models and enter verified current USD/million rates.
# Each model makes len(cases) paid calls. Rates are optional: unknown is shown explicitly.
model_configs = [
    {"model": MODEL, "rates": None},
    # {"model": "your-second-accessible-model", "rates": (input_rate, output_rate)},
]
def compare_candidates(configs, examples):
    comparison = {}
    for config in configs:
        print("\nModel:", config["model"])
        rows = evaluate_model(config["model"], examples, config["rates"])
        comparison[config["model"]] = rows
        summarize_evaluation(rows)
    return comparison
# comparison = compare_candidates(model_configs, cases[:3])  # expand to all 20 after a small run


**Decision record:** Task = support classification; dataset = 20 synthetic examples; target = your chosen quality, cost and latency thresholds. Compare at least two accessible models on the **same** examples and prompt, record model IDs, date and rates, and write a recommendation. Treat failures, refusals and incomplete responses as failures; inspect them separately. Check the actual provider invoice/usage for exact cost. Tiny datasets give uncertain estimates and do not establish production performance.


## Hands-on exercises

Work out each answer before running the code cells.

1. **Cost:** One conversation uses 500 system + 200 question + 2,000 retrieved input tokens and 500 output tokens. Estimate tokens per conversation, per day for 1,000 conversations, and daily/monthly cost using hypothetical $5 input / $15 output per million.
2. **Model selection:** A: 96% accuracy and $0.12/review; B: 88% and $0.02; C: 94% and $0.08. Requirement: >95% and <$0.05. Which qualifies?
3. **Context:** Allocate 8,000 tokens across instructions, user question, relevant documents, output and safety margin. Explain your trade-offs.
4. **Failure modes:** How would you verify a historical weather fact, translate a 50,000-word document, or count many tiny objects in an image?


In [ ]:
# Offline: exercise 1
input_tokens = 500 + 200 + 2000
output_tokens = 500
per_conversation = cost_usd(input_tokens, output_tokens, 5, 15)
print("Tokens/conversation:", input_tokens+output_tokens,
      "| tokens/day:", (input_tokens+output_tokens)*1000,
      "| daily USD:", per_conversation*1000,
      "| 30-day USD:", per_conversation*1000*30)
assert abs(per_conversation*1000 - 21) < 1e-9


**Checks:** (1) 3,200 tokens/conversation, 3.2 million/day, $21/day and $630/30 days at the hypothetical rates. (2) None meets both constraints; change a requirement or improve and retest. (3) Many allocations work; reserve output as well as buffer, as shown in the context-budget cell. (4) Use a trusted weather source; chunk and verify translation with a terminology guide; use suitable counting software and human review where needed.


## Practical assessment and self-check

Run a model-selection project using the 20 examples above or replace them with your own labeled, non-sensitive dataset:

1. State the exact task and acceptance thresholds before testing.
2. Compare 2–3 accessible models on the same 15–20 examples. Start with three examples to catch setup errors.
3. Report correct/total, failed/invalid counts, latency, token use, estimated cost and any unknown costs. Inspect each failure.
4. Write a paragraph naming your choice, reasons, trade-offs, known limitations and what additional tests you would run.
5. Create a held-out set and repeat after any prompt or model change.

**Questions to answer without looking back:** How does next-token generation work? What is the difference between context, training knowledge and application memory? What changes when streaming? Why can structured-looking output still be wrong? How do you choose a model when quality, latency and cost disagree?

### References

- [OpenAI Python API library](https://developers.openai.com/api/reference/python) and [models](https://developers.openai.com/api/docs/models)
- [Pricing](https://developers.openai.com/api/docs/pricing), [streaming](https://developers.openai.com/api/docs/guides/streaming-responses), [images](https://developers.openai.com/api/docs/guides/images-vision), [file inputs](https://developers.openai.com/api/docs/guides/file-inputs), [speech-to-text](https://developers.openai.com/api/docs/guides/speech-to-text)
- [OpenAI tiktoken](https://github.com/openai/tiktoken)
